In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", "..", ".."))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path

'/root/Workspace/quotaclimat'

In [ ]:
from quotaclimat.data_ingestion.advertising_detection.e02_create_chunks import Chunk
from quotaclimat.data_ingestion.advertising_detection.e04_group_chunks import (
    ChunkGrouping,
    debug_pair
)
from quotaclimat.data_ingestion.advertising_detection.tools.cache import LocalCache
from quotaclimat.data_ingestion.advertising_detection.e00_partition_window import (
    partition_week_program,
)

import json
from datetime import timedelta
from pathlib import Path

In [ ]:
channel = "tf1"
start_date = "2025-05-05"

partition = partition_week_program(
    channel=channel,
    start_date=start_date,
    margin=timedelta(minutes=15),
)

with LocalCache(name="chunks", version="8f4b4c10c550365f", cache_folder=Path("./../../../../.cache")) as chunk_cache:
    chunks: list[Chunk] = []
    for segment in partition:
        chunk_batch = json.loads(chunk_cache.get(segment.identifier + ".json"))
        chunks.extend([Chunk.from_dict(d) for d in chunk_batch])

FileNotFoundError: [Errno 2] No such file or directory: '.cache/chunks/8f4b4c10c550365f/tf1_2025-05-05_06-40-00.json'

In [ ]:
chunk_a = chunks.find(lambda c: abs(c.start_sec - 1746420874.53) < 1)
chunk_b = chunks.find(lambda c: abs(c.start_sec - 1746680080.76) < 1)

debug_pair(
    chunk_a,
    chunk_b,
    ChunkGrouping(
        similarity_threshold=0.05,
        min_matching_hashes=1,
        duration_tol=1.5,
        rms_tol=0.05,
        centroid_tol=0.05,
        zcr_tol=0.1,
    ),
)


DEBUG: chunk pair grouping analysis
  A: [1746600703.61s – 1746600715.85s]  channel=tf1
  B: [1746516838.20s – 1746516849.37s]  channel=tf1

[1] Minimum duration filter (>= 0.5 s)
    A duration: 12.237s  ✓
    B duration: 11.169s  ✓

[2] Acoustic pre-filter (_features_compatible)
    duration |A-B| = 1.068s  (tol=1.5)  ✓
    energy_mean rel_diff = 0.0859  (tol=0.05)  ✗  (A=0.0354, B=0.0387)
    spectral_centroid rel_diff = 0.0050  (tol=0.05)  ✓  (A=2795.6, B=2809.6)
    zcr_mean rel_diff = 0.0072  (tol=0.1)  ✓  (A=0.1287, B=0.1296)
  → BLOCKED: acoustic pre-filter rejected this pair.
